In [ ]:
import easysteer.hidden_states as hs
import torch
from vllm import LLM

model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

llm = LLM(
    model=model_name,   # Model path
    tensor_parallel_size=1,
    enforce_eager=True,
    enable_chunked_prefill=False, # Hidden states extraction doesn't support prefix caching yet
    enable_prefix_caching=False   # Hidden states extraction doesn't support chunked prefill yet
)

# Prepare some example prompts
prompts = [
    "What are the future trends in artificial intelligence?",
    "Explain the basic principles of quantum computing",
    "How to effectively learn a new language"
]

# Extract hidden states for all tokens in the prompts
batch_hidden_states, outputs = hs.get_all_hidden_states_generate(llm, prompts)

# Get and keep the hidden states of the last token for each prompt
all_hidden_states = []
batch_last_token_hidden_states = []
for hidden_state in batch_hidden_states:
    hidden_state_by_layers = []
    for layer in hidden_state:
        hidden_state_by_layers.append(layer[-1, :])
    all_hidden_states.append(torch.stack(hidden_state_by_layers))
all_hidden_states = torch.stack(all_hidden_states)
torch.save(all_hidden_states.detach().cpu(), "qwen2_5VL_all_hidden_states_tensor.pt")